In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import shutil
import torch.nn as nn
import os
import pandas as pd
import random

from PatchDataset import load_dataset

from torch.utils.data import DataLoader,Subset,Dataset
from torchvision import datasets, transforms
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score

In [3]:
PATCH_SIZE = 224
BATCH_SIZE = 32
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


PATCHES_ROOT = "patches_dataset/patches_v1_seed42"
csv_path = os.path.join(PATCHES_ROOT, "metadata.csv")


In [ ]:
# loading the dataset
df = pd.read_csv(csv_path)
df["patch_id"] = df["patch_id"].astype(int)
df["label"] = df["label"].astype(int)
df["type"] = df["type"].astype(str)

train_dataset,test_dataset,train_loader,test_loader = load_dataset(df,PATCHES_ROOT,BATCH_SIZE)

for xb,yb in train_loader:
    print(xb.shape,yb.shape,min(yb),max(yb))
    break

train/test: 7056 1764
torch.Size([32, 3, 224, 224]) torch.Size([32]) tensor(0.) tensor(1.)


In [ ]:
from transformers import CLIPProcessor, CLIPModel

model_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor_clip = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")



config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
# some testing individual images

#imp = "patches_dataset/patches_v1_seed42/negative/817.png"
#imp = "patches_dataset/patches_v1_seed42/negative/0.png"
#imp = "patches_dataset/patches_v1_seed42/negative/49.png"
imp = "patches_dataset/patches_v1_seed42/negative/0.png"
image = Image.open(imp)
labels = [
    #"a tree",
    #"not a tree"
    "an aerial image of a patch of land without a tree",
    "an aerial image of a patch of land with a tree",
    "an aerial image of a house",
    "an areal image of a branch on the ground"
]

inputs = processor_clip(text=labels, images=image, return_tensors="pt", padding=True)

outputs = model_clip(**inputs)
logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)
most_likely_idx = probs.argmax(dim=1).item()
most_likely_label = labels[most_likely_idx]
print(f"Most likely label: {most_likely_label} with probability: {probs[0][most_likely_idx].item():.3f}")


Most likely label: an aerial image of a patch of land without a tree with probability: 0.634


In [ ]:
prompt_sets = {
    "simple": ["tree", "no tree"],
    "article": [
        "an image of a tree", 
        "an image of land without trees"
    ],
    "aerial": [
        "aerial view tree patch",
        "aerial view empty land"
    ],
    "detailed": [
        "aerial forest patch with tree crown and branches", 
        "aerial forest patch with bare ground and no vegetation"
    ],
    "negative": [
        "patch with single tree",
        "patch with no trees no vegetation no branches"
    ],
    "expert": [
        "aerial orthophoto with tree canopy", 
        "aerial orthophoto barren forest floor"
    ]
}

In [ ]:
def clip_auc_on_loader(model, processor, loader, prompts, device, max_batches=100):
    model.eval()
    all_pos_scores = []
    all_labels = []
    
    with torch.no_grad():
        for i, (images, labels) in enumerate(tqdm(loader)):
            if i >= max_batches: break
            
            images = images.to(device)
            
            # Process with CLIP processor (handles tensor -> PIL internally)
            inputs = processor(
                text=prompts, 
                images=images,  
                return_tensors="pt", 
                padding=True
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image  # (B, len(prompts))
            
            # Positive class score = softmax of tree prompt(s)
            tree_logits = logits_per_image[:, 1]  # assume prompts[1] = tree
            pos_probs = torch.softmax(tree_logits, dim=0)
            
            all_pos_scores.extend(pos_probs.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    y_score = np.array(all_pos_scores)
    y_true = np.array(all_labels)
    auroc = roc_auc_score(y_true, y_score)
    return auroc

device = "cuda" if torch.cuda.is_available() else "cpu"
model_clip = model_clip.to(device)
processor_clip = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

results = {}
for name, prompts in prompt_sets.items():
    auc = clip_auc_on_loader(model_clip, processor_clip, test_loader, prompts, device)
    results[name] = auc
    print(f"{name:20}: AUROC {auc:.4f}")


100%|██████████| 56/56 [03:42<00:00,  3.97s/it]


simple              : AUROC 0.5860


100%|██████████| 56/56 [03:45<00:00,  4.02s/it]


article             : AUROC 0.6359


100%|██████████| 56/56 [10:36<00:00, 11.36s/it]   


...

detailed            : AUROC 0.6127


100%|██████████| 56/56 [03:43<00:00,  3.99s/it]


negative            : AUROC 0.4892


100%|██████████| 56/56 [03:42<00:00,  3.98s/it]

expert              : AUROC 0.5678
